# onnx runtime vs pytorch eager

exporting a small resnet to onnx and timing inference under both runtimes. trying to decide if it is worth the export friction for a serving stack.

In [ ]:
import torch
import torchvision.models as tvm

model = tvm.resnet18(weights=None).eval()
x = torch.randn(1, 3, 224, 224)

# export
torch.onnx.export(model, x, 'rn18.onnx',
                  input_names=['x'], output_names=['logits'],
                  opset_version=14)


In [ ]:
# pip install onnxruntime==1.12
import onnxruntime as ort
import numpy as np
import time

sess = ort.InferenceSession('rn18.onnx', providers=['CPUExecutionProvider'])

def bench(fn, iters=20):
    fn()  # warm
    t0 = time.time()
    for _ in range(iters):
        fn()
    return (time.time() - t0) / iters

x_np = x.numpy()

t_ort = bench(lambda: sess.run(['logits'], {'x': x_np}))
with torch.no_grad():
    t_pt = bench(lambda: model(x))
print('onnxruntime:', t_ort)
print('pytorch eager:', t_pt)


rough numbers on my laptop cpu: onnxruntime ~30-40 percent faster for this model. enough to take, but the export step is brittle for anything with control flow.

rerun on torch 1.12 cpu, same numbers within noise.

In [ ]:
# add weight decay
# opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)


tried bumping batch size, no real change.

In [ ]:
import torch
torch.manual_seed(0)
